<a href="https://colab.research.google.com/github/dewi-ikrimah/pm-turi2-prepocessing-Dewi-Ikrimah/blob/main/Copy_of_PM_P4_Dewi_Ikrimah_2488010049.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import numpy as np



# Dataset karyawan (sengaja mengandung masalah kualitas)
data = {
    'usia'      : [25, 32, np.nan, 45, 28, 51, 38, np.nan, 29, 41],
    'pendapatan': [4_000_000, 7_500_000, 5_200_000, 12_000_000, 4_800_000, 15_000_000, 8_100_000, 6_300_000, np.nan, 9_900_000],
    'pendidikan': ['SMA','S1','SMA','S2','SMA','S2','S1','S1','SMA','S2'],  # ordinal
    'kota'      : ['Bandung','Jakarta','Bandung','Surabaya','Jakarta','Surabaya','Jakarta','Bandung','Bandung','Jakarta'],    # nominal
    'membeli'   : ['Tidak','Ya','Tidak','Ya','Tidak','Ya','Ya','Tidak','Tidak','Ya']  # target
}
df = pd.DataFrame(data)
df

# Langkah 1 — Menangani nilai hilang (imputasi mean).
df['usia'] = df['usia'].fillna(df['usia'].mean())
df['pendapatan'] = df['pendapatan'].fillna(df['pendapatan'].median())
df['status'] = ['Tetap', 'Kontrak', 'Kontrak', 'Tetap', 'Kontrak', 'Tetap', 'Tetap', 'Kontrak', 'Tetap', 'Kontrak']

# Langkah 2 — Memisahkan fitur (X) dan label (y).
X = df.drop(columns=['membeli'])
y = df['membeli']

# Langkah 3 — Encoding kategorikal (ordinal & one-hot).
X['pendidikan'] = X['pendidikan'].map({'SMA':0, 'S1':1, 'S2':2})    # ordinal
X = pd.get_dummies(X, columns=['kota', 'status'], dtype=int) # nominal
y = y.map({'Tidak':0,'Ya':1})

# Langkah 4 — Membagi data (SEBELUM scaling).
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Langkah 5 — Penskalaan (fit pada latih, transform pada latih & uji).
from sklearn.preprocessing import MinMaxScaler
num = ['usia', 'pendapatan', 'pendidikan']
sc = MinMaxScaler()
X_train[num] = sc.fit_transform(X_train[num])
X_test[num] = sc.transform(X_test[num]) # cegah data leakage

# Langkah 6 — Verifikasi: tidak ada nilai hilang, semua numerik, skala seragam.
X_train.describe()


,usia,pendapatan,pendidikan,kota_Bandung,kota_Jakarta,kota_Surabaya,status_Kontrak,status_Tetap
count,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000
mean,0.530357,0.398214,0.428571,0.428571,0.428571,0.142857,0.571429,0.428571
std,0.349971,0.367808,0.449868,0.534522,0.534522,0.377964,0.534522,0.534522
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.353125,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.556250,0.287500,0.500000,0.000000,0.000000,0.000000,1.000000,0.000000
75%,0.725000,0.625000,0.750000,1.000000,1.000000,0.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Jelaskan mengapa fit_transform pada latih tetapi hanya transform pada uji.

Penggunaan .fit_transform() pada data latih bertujuan agar scaler memperlajari patokan statistik data latih (seperti min dan max) sekaligus mengubah skalanya. Sementara itu, data uji hanya menggunakan .transform() agar diubah skalanya mengikuti patokan dari data latih tadid tanpa menghitung statistik baru, sehingga mencegah kebocoran daya (data leakage) dan menjaga evaluasi model tetap jujur.

1. Mengapa urutan 'split dulu, baru scaling' penting?

Karena jika scaler di fit pada seluruh data sebelum split, maka informasi data uji 'bocor' ke pelatihan. Akibatnya, evaluasi menjadi tidak jujur, model tampak lebih baik dari pada kenyataannya.

2. Kapan label encoding dan kapan one-hot encoding?

**Label encoding** digunakan ketika untuk data yang ordinal (berurutan), yaitu data kategori yang memiliki urutan, tingkatan, atau hirarki yang jelas. Contohnya, tingkat pendidikan, ukuran pakaian, dan tingkat kepuasan.
Sementara **one-hot encoding** digunakan untuk variabel kategorikal nominal, yaitu data kategori yang tidak memiliki urutan atau hirarki. Mengubah satu kolom menjadi beberapa kolom biner baru (0/1). Contohnya, nama kota, status kepegawaian.

3. Apa perbedaan normalisasi dan standardisasi?

**Normaslisasi** adalah mengubah data numerik ke dalam rentang skala 0-1, berguna bila butuh batas yang jelas, dan sensitif terhadap outlier. Sementara **Standarisasi** mengandalkan nilai rata-rata dan standar deviasi, sehingga data dipusatkan ke mean 0 dengan standar deviasi 1, tidak membatasi rentang, dan lebih tahan outlier.